# পাঠ ১০ - উৎপাদনে AI এজেন্টস

এই পাঠে আপনি Microsoft Agent Framework (Python) ব্যবহার করে AI এজেন্টসের জন্য **উৎপাদন প্যাটার্নসমূহ** শیکھবেন। আমরা আলোচনা করব:

- **পর্যবেক্ষণযোগ্যতা** — এজেন্টের পারস্পরিক ক্রিয়ায় সময় এবং লগিং যোগ করা
- **মূল্যায়ন** — একটি মূল্যায়নকারী এজেন্টের মাধ্যমে প্রতিক্রিয়ার গুণমান স্কোর করা
- **ব্যয় ব্যবস্থাপনা** — টোকেন অপ্টিমাইজেশন এবং মডেল নির্বাচন কৌশল

এই পরিস্থিতিটি একটি **ভ্রমণ এজেন্ট** যারা ব্যবহারকারীদের ট্রিপ পরিকল্পনায় সাহায্য করে, যার ওপর পর্যবেক্ষণ এবং মূল্যায়ন স্তরযুক্ত। 


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## উৎপাদন বিবেচনা

AI এজেন্টদের প্রোটোটাইপ থেকে উৎপাদনে নিয়ে যাওয়া তিনটি মূল স্তম্ভের প্রতি যত্নশীল মনোযোগ দাবি করে:

১. **পর্যবেক্ষণযোগ্যতা** — এজেন্ট কী করে, কত সময় লাগে এবং কোন টুলগুলি কল করে তা দেখতে হবে। ট্রেসিং ও লগিং ছাড়া উৎপাদন সংক্রান্ত সমস্যা ডিবাগ করা প্রায় অসম্ভব।

২. **মূল্যায়ন** — স্বয়ংক্রিয় গুণগত মান পরীক্ষা নিশ্চিত করে যে এজেন্টের প্রতিক্রিয়াগুলি সময়ের সাথে সঠিক, সম্পূর্ণ এবং সহায়ক থাকে। একটি মূল্যায়নকারী এজেন্ট সংজ্ঞায়িত মানদণ্ডের বিরুদ্ধে প্রতিক্রিয়া প্রদান করতে পারে।

৩. **ব্যয় ব্যবস্থাপনা** — টোকেন ব্যবহারের সরাসরি প্রভাব খরচে পড়ে। প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন এবং ক্যাশিং-এর মতো কৌশলগুলি ব্যয় নিয়ন্ত্রণে রাখতে সাহায্য করে, গুণমানের কোনো ক্ষতি ছাড়াই।


## একটি অবজারভেবল এজেন্ট তৈরি করা

আমরা ভ্রমন সরঞ্জামগুলি সংজ্ঞায়িত করি এবং এজেন্ট কলটি টাইমিং-এর সাথে মোড়ানো হয় যাতে আমরা বাকগতিকে পর্যবেক্ষণ করতে পারি। প্রোডাকশনে আপনি OpenTelemetry বা একটি সমতুল্য ট্রেসিং ব্যাকএন্ডের সাথে ইন্টিগ্রেট করবেন।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## মূল্যায়ন প্যাটার্নসমূহ

একটি সাধারণ প্রোডাকশন প্যাটার্ন হল দ্বিতীয় একটি এজেন্টকে **মূল্যায়নকারী** হিসেবে ব্যবহার করা। মূল্যায়নকারী প্রাথমিক এজেন্টের প্রতিক্রিয়াকে পূর্বনির্ধারিত মানদণ্ড যেমন সম্পূর্ণতা, সঠিকতা, এবং সহায়কতা অনুসারে স্কোর করে।

এটি সক্ষম করে:
- ব্যবহারকারীর কাছে প্রতিক্রিয়া পৌঁছানো আগের স্বয়ংক্রিয় গুণমান নিয়ন্ত্রণ
- প্রম্পট বা মডেল পরিবর্তিত হলে রিগ্রেশন সনাক্তকরণ
- সময়ের সাথে এজেন্টের পারফরম্যান্সের ক্রমাগত পর্যবেক্ষণ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## খরচ ব্যবস্থাপনার কৌশল

উৎপাদন AI এজেন্টগুলির জন্য খরচ নিয়ন্ত্রণ করা গুরুত্বপূর্ণ। এখানে প্রধান কৌশলগুলি রয়েছে:

| কৌশল | বিবরণ |
|---|---|
| **প্রম্পট অপ্টিমাইজেশন** | সিস্টেম নির্দেশাবলী সংক্ষিপ্ত রাখুন। ইনপুট টোকেন কমানোর জন্য অতিরিক্ত প্রসঙ্গ বাদ দিন। |
    "| **মডেল নির্বাচন** | সহজ কাজ যেমন শ্রেণীবিভাগ বা নিষ্কাশনের জন্য ছোট, সস্তা মডেল (যেমন GPT-4.1-mini) ব্যবহার করুন, এবং জটিল যুক্তির জন্য বড় মডেল সংরক্ষণ করুন। |\n",
| **ক্যাশিং** | পুনরাবৃত্ত API কল এড়াতে টুল ফলাফল এবং ঘনঘন করা অনুসন্ধান ক্যাশ করুন। |
| **টোকেন বাজেট** | প্রত্যাশিত চেয়ে দীর্ঘ প্রতিক্রিয়া প্রতিরোধে `max_tokens` সীমা নির্ধারণ করুন। |
| **ব্যাচিং** | যেখানে সম্ভব একক API কল-এ একাধিক ব্যবহারকারীর অনুরোধ গুচ্ছবদ্ধ করুন। |

বাস্তবে, একটি স্তরভিত্তিক পদ্ধতি ভাল কাজ করে: সরল অনুরোধগুলি দ্রুত, সস্তা মডেলে পাঠান এবং শুধুমাত্র জটিল অনুসন্ধানগুলি আরও সক্ষম মডেলে উন্নয়ন করুন।


## সারসংক্ষেপ

এই পাঠে আপনি শিখলেন কীভাবে:

১। এজেন্টের ইন্টারঅ্যাকশনে **পর্যবেক্ষণ যোগ করুন** সময় নির্ধারণ এবং লগিং-এর মাধ্যমে, যা ট্রেসিং এবং মনিটরিং-এর ভিত্তি গড়ে তোলে।
২। **এজেন্টের প্রতিক্রিয়া স্বয়ংক্রিয়ভাবে মূল্যায়ন করুন** একটি ইভ্যালুয়েটর এজেন্ট ব্যবহার করে যা সম্পূর্ণতা, সঠিকতা এবং সহায়কতা স্কোর দেয়।
৩। **খরচ পরিচালনা করুন** প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন, ক্যাশিং এবং টোকেন বাজেটের মাধ্যমে।

এই প্রোডাকশন প্যাটার্নগুলি নিশ্চিত করে আপনার AI এজেন্টগুলি নির্ভরযোগ্য, পরিমাপযোগ্য এবং ব্যাপকভাবে খরচ-সাশ্রয়ী। 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
